In [3]:
!pip install pymongo
import pandas as pd
from pymongo import MongoClient

In [4]:
# Connect to local MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["tmdb_movies"]
print("Connected to MongoDB!")

# Load the cleaned CSV
df = pd.read_csv("TMDB_cleaned.csv")
df = df.where(pd.notnull(df), None)
print(f"Loaded {len(df)} rows")

Connected to MongoDB!
Loaded 24985 rows


In [5]:
# Convert dataframe to list of dictionaries
movies = df.to_dict(orient='records')

# Insert into MongoDB collection
collection = db["movies"]
collection.drop()
collection.insert_many(movies)

print(f"Inserted {collection.count_documents({})} documents into MongoDB!")

Inserted 24985 documents into MongoDB!


In [ ]:
# ── QUERY 1: Movies with highest audience engagement by genre ──
print("\n--- Query 1: Top 5 Most Voted Movies per Genre ---")
pipeline = [
    { "$match": { "vote_count": { "$gte": 1000 } } },
    { "$unwind": "$genres" },
    { "$sort": { "vote_count": -1 } },
    { "$group": {
        "_id": "$genres",
        "top_movie": { "$first": "$title" },
        "vote_count": { "$first": "$vote_count" },
        "revenue": { "$first": "$revenue" }
    }},
    { "$sort": { "vote_count": -1 } },
    { "$limit": 5 }
]
results1 = list(collection.aggregate(pipeline))
for r in results1:
    print(r)

# ── QUERY 2: Average revenue by genre for movies with budget > 1M ──
print("\n--- Query 2: Average Revenue by Genre (budget > $1M) ---")
pipeline2 = [
    { "$match": { "budget": { "$gt": 1000000 } } },
    { "$unwind": "$genres" },
    { "$group": {
        "_id": "$genres",
        "avg_revenue": { "$avg": "$revenue" },
        "avg_budget": { "$avg": "$budget" },
        "movie_count": { "$sum": 1 }
    }},
    { "$sort": { "avg_revenue": -1 } },
    { "$limit": 10 }
]
results2 = list(collection.aggregate(pipeline2))
for r in results2:
    print(r)